# Gemini version of 04c — cosine similarity to self (within-subject)

Parallel to `04c_draw_generate_cossim_toself`, using the Gemini full-drawing embeddings
from `03g_draw_generate_gemini_embedding`. Same within-subject 5-way (own category
excluded, since each participant draws each category once) softmax logic:

Softmax uses `T = 1` (plain `softmax(sims)`), as the CLIP pipeline does. This is a choice, and we could also consider adjusting the temperature to make Gemini and CLIP spread more comprable. Gemini's
within-participant cosine spread is narrow, so the resulting probabilities are close to
uniform (~0.20).

Output: `data/preprocessed/gemini/draw_cossim_toself.csv`, same `draw_cossim_{cat}` /
`draw_probability_{cat}` columns as the CLIP version.

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]

EMBED_CSV = "../data/embeddings/gemini/draw_gemini_embeddings_full.csv"
OUT_CSV = "../data/preprocessed/gemini/draw_cossim_toself.csv"

FLAGS_CSV = "../data/tidy/drawings.csv"  # interference flags

# Interference trials are excluded throughout, so they enter neither the comparison set nor the output
_flags = pd.read_csv(FLAGS_CSV)
INTERFERENCE = set(zip(_flags.loc[_flags["draw_interference"] == 1, "participant_id"].str.strip(),
                       _flags.loc[_flags["draw_interference"] == 1, "target_category"].str.strip().str.lower()))

In [2]:
def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x, dtype=float)
    if pd.isna(x):
        return np.nan
    return np.array(ast.literal_eval(x), dtype=float)


def cosine_sim_to_matrix(query, matrix):
    """cosine(query, each row of matrix) -> 1D array, len == matrix.shape[0]."""
    q = query / np.linalg.norm(query)
    m = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    return m @ q


def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

In [3]:
draw_embeddings = pd.read_csv(EMBED_CSV)
draw_embeddings["embedding"] = draw_embeddings["embedding"].apply(parse_embedding)
draw_embeddings["target_category"] = draw_embeddings["target_category"].str.lower().str.strip()
draw_embeddings = draw_embeddings[[k not in INTERFERENCE for k in zip(draw_embeddings["participant_id"], draw_embeddings["target_category"])]].copy()
print(draw_embeddings.shape)
draw_embeddings.head()

(610, 6)


,participant_id,age_months,target_category,image_path,is_adult,embedding
0,AD001,229.0,seahorse,../data/raw/mongo_output/sketches_full_dataset...,True,"[-0.05214051529765129, -0.02649662084877491, 0..."
1,AD001,229.0,octopus,../data/raw/mongo_output/sketches_full_dataset...,True,"[-0.05687795951962471, -0.009877415373921394, ..."
2,AD001,229.0,crab,../data/raw/mongo_output/sketches_full_dataset...,True,"[-0.033965226262807846, -0.024662217125296593,..."
3,AD001,229.0,turtle,../data/raw/mongo_output/sketches_full_dataset...,True,"[-0.04598578065633774, -0.004204542841762304, ..."
4,AD001,229.0,shark,../data/raw/mongo_output/sketches_full_dataset...,True,"[-0.04494376480579376, -0.029236141592264175, ..."


## Softmax temperature

`T = 1`, matching the CLIP branch (plain `softmax(sims)`). Again, this is a choice but I think most comprable to the CLIP pipeline.

In [4]:
emb = {(r["participant_id"], r["target_category"]): r["embedding"]
       for _, r in draw_embeddings.iterrows()}

spreads = []
for pid in draw_embeddings["participant_id"].unique():
    vecs = {c: emb[(pid, c)] for c in SEA_ANIMALS if (pid, c) in emb}
    if len(vecs) < 3:
        continue
    for c_query in vecs:
        others = [c for c in vecs if c != c_query]
        sims = [float(np.dot(vecs[c_query], vecs[c]) /
                       (np.linalg.norm(vecs[c_query]) * np.linalg.norm(vecs[c])))
                for c in others]
        spreads.append(max(sims) - min(sims))

spreads = np.array(spreads)
median_spread = float(np.median(spreads))
TEMPERATURE = 1.0  # same as the CLIP branch, which uses plain softmax(sims)

print(f"within-participant cosine spread (n={len(spreads)}): median {median_spread:.4f}")
print(f"softmax temperature: {TEMPERATURE:.4f}  (i.e. softmax(sims / {TEMPERATURE:.4f}))")

within-participant cosine spread (n=607): median 0.0778
softmax temperature: 1.0000  (i.e. softmax(sims / 1.0000))


In [5]:
def add_within_subject_draw_similarities(
    draw_embeddings,
    temperature=1.0,
    embedding_col="embedding",
    participant_col="participant_id",
    category_col="target_category",
    categories=None,
):
    if categories is None:
        categories = SEA_ANIMALS

    draw_embeddings = draw_embeddings.copy()

    for cat in categories:
        draw_embeddings[f"draw_cossim_{cat}"] = np.nan
        draw_embeddings[f"draw_probability_{cat}"] = np.nan

    for pid, participant_df in draw_embeddings.groupby(participant_col):
        for idx, row in participant_df.iterrows():
            query_emb = row[embedding_col]
            if not isinstance(query_emb, np.ndarray):
                continue

            comparison_categories = []
            comparison_embeddings = []
            for cat in categories:
                cat_rows = participant_df[participant_df[category_col] == cat]
                cat_embeddings = [
                    emb for row_idx, emb in zip(cat_rows.index, cat_rows[embedding_col])
                    if isinstance(emb, np.ndarray) and row_idx != idx
                ]
                if len(cat_embeddings) == 0:
                    continue
                category_mean = np.mean(np.stack(cat_embeddings), axis=0)
                comparison_categories.append(cat)
                comparison_embeddings.append(category_mean)

            if len(comparison_embeddings) == 0:
                continue

            category_matrix = np.stack(comparison_embeddings)
            sims = cosine_sim_to_matrix(query_emb, category_matrix)
            probs = softmax(sims / temperature)

            for cat, sim, prob in zip(comparison_categories, sims, probs):
                draw_embeddings.loc[idx, f"draw_cossim_{cat}"] = sim
                draw_embeddings.loc[idx, f"draw_probability_{cat}"] = prob

    return draw_embeddings

In [6]:
draw_embeddings_with_sims = add_within_subject_draw_similarities(
    draw_embeddings, temperature=TEMPERATURE
)

prob_cols = [f"draw_probability_{cat}" for cat in SEA_ANIMALS]
draw_embeddings_with_sims["prob_sum"] = (
    draw_embeddings_with_sims[prob_cols].sum(axis=1, min_count=1)
)
draw_embeddings_with_sims["prob_sum"].describe()

count    6.070000e+02
mean     1.000000e+00
std      8.461459e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: prob_sum, dtype: float64

In [7]:
cols_to_drop = ["embedding", "prob_sum"]
result = draw_embeddings_with_sims.drop(columns=[c for c in cols_to_drop if c in draw_embeddings_with_sims.columns])

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"Saved {result.shape[0]} rows -> {OUT_CSV}")
print(result.columns.tolist())

Saved 610 rows -> ../data/preprocessed/gemini/draw_cossim_toself.csv
['participant_id', 'age_months', 'target_category', 'image_path', 'is_adult', 'draw_cossim_crab', 'draw_probability_crab', 'draw_cossim_octopus', 'draw_probability_octopus', 'draw_cossim_seahorse', 'draw_probability_seahorse', 'draw_cossim_shark', 'draw_probability_shark', 'draw_cossim_turtle', 'draw_probability_turtle', 'draw_cossim_whale', 'draw_probability_whale']
